# DuckDB UI notebook JSON exporter

Export latest DuckDB UI notebook JSON payloads with one cell object per line for cleaner diffs.

**NOTE:** Close DuckDB UI before running this notebook if DuckDB reports that `ui.db` is locked.

In [1]:
from pathlib import Path

# DuckDB UI notebook database.
UI_DB_PATH = Path.home() / ".duckdb" / "extension_data" / "ui" / "ui.db"

# Folder where .json exports will be saved.
EXPORT_DIR = Path(r"D:\Codex\Python-DuckDB_notebook_exporter\json")

# Notebook titles to export.
NOTEBOOKS_TO_EXPORT = []  # empty list will export ALL notebooks
NOTEBOOKS_TO_EXPORT = ["Olympics", "Billboard Hot 100"]

In [3]:
import json
import re
from collections import OrderedDict

import duckdb
import pandas as pd
from IPython.display import display


def safe_filename(value: str, max_length: int = 180) -> str:
    """Return a Windows-safe filename stem."""
    cleaned = re.sub(r'[<>:"/\\|?*\x00-\x1F]', "_", value).rstrip(" .")
    return (cleaned[:max_length] or "notebook")


def ordered_cell(cell: dict) -> OrderedDict:
    """Put compact cell metadata before the long query text."""
    ordered = OrderedDict()

    for key in ("cellId", "useDatabase", "query"):
        if key in cell:
            ordered[key] = cell[key]

    for key, value in cell.items():
        if key not in ordered:
            ordered[key] = value

    return ordered


def notebook_json_for_diff(payload: str) -> tuple[str, int]:
    notebook = json.loads(payload, object_pairs_hook=OrderedDict)
    cells = [ordered_cell(cell) for cell in notebook.get("cells", [])]

    rest = OrderedDict((key, value) for key, value in notebook.items() if key != "cells")
    cell_lines = [json.dumps(cell, ensure_ascii=False, separators=(",", ":")) for cell in cells]
    rest_json = json.dumps(rest, ensure_ascii=False, separators=(",", ":"))[1:]

    if cell_lines:
        text = '{"cells":[\n' + ",\n".join(cell_lines) + "\n]," + rest_json
    else:
        text = '{"cells":[],' + rest_json

    return text, len(cells)


latest_notebooks_query = """
SELECT notebook_id, title, version, json, created
FROM notebook_versions
WHERE expires IS NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY title ORDER BY version DESC, created DESC) = 1
ORDER BY title;
"""

with duckdb.connect(str(UI_DB_PATH), read_only=True) as con:
    notebooks_df = con.sql(latest_notebooks_query).df()

if NOTEBOOKS_TO_EXPORT:
    notebooks_df = notebooks_df[notebooks_df["title"].isin(NOTEBOOKS_TO_EXPORT)].copy()

display(notebooks_df.drop(columns=["json"]).head(5))

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

exports = []

for row in notebooks_df.itertuples(index=False):
    export_json, cell_count = notebook_json_for_diff(row.json)
    export_path = EXPORT_DIR / f"{safe_filename(row.title)}.json"
    export_path.write_text(export_json + "\n", encoding="utf-8")

    exports.append(
        {
            "title": row.title,
            "version": row.version,
            "cell_count": cell_count,
            "export_path": str(export_path),
            "bytes": export_path.stat().st_size,
        }
    )

exports_df = pd.DataFrame(exports)
display(exports_df.head(5))

,notebook_id,title,version,created
0,c07f9e6e-baae-4958-aecf-64ffb2e5c42b,Act I - backblaze,144,2026-04-21 21:44:43.536468
1,586c7c76-b423-4bc0-b44b-813960e257dc,Act II - portability,52,2026-04-21 21:59:20.030836
2,f1377341-fb8c-4c4a-b8a4-3f82c0217955,Act III baby names,254,2026-06-04 19:32:54.790358
3,5ade5e03-a647-4913-a95e-6197f8dc0782,Billboard Hot 100,35,2026-06-07 03:23:51.531642
4,543143fb-1e38-4af9-a05d-0f78195a08f2,Bookmarks,112,2026-03-20 21:11:13.844804


,title,version,cell_count,export_path,bytes
0,Act I - backblaze,144,28,D:\Codex\Python-DuckDB_notebook_exporter\json\...,12296
1,Act II - portability,52,12,D:\Codex\Python-DuckDB_notebook_exporter\json\...,2937
2,Act III baby names,254,17,D:\Codex\Python-DuckDB_notebook_exporter\json\...,9791
3,Billboard Hot 100,35,4,D:\Codex\Python-DuckDB_notebook_exporter\json\...,732
4,Bookmarks,112,6,D:\Codex\Python-DuckDB_notebook_exporter\json\...,1493
